# 第 5 章补充：SARSA——按当前策略学习动作价值

这一课只回答一个问题：**若智能体实际带着探索在玩游戏，怎样评价它当前这套真实玩法中的每个动作？** 答案是 SARSA。读完后应能写出 SARSA 五元组、更新表格，并说明它为什么属于 TD 而不是蒙特卡洛方法。


## 1. SARSA 想学的是“当前这套玩法”的评分

SARSA 不是直接学习理想状态下的最优动作价值 $Q_\star$；它学习的是当前策略 $\pi$ 的动作价值：

$$
Q_\pi(s,a).
$$

人话：在状态 $s$ 做动作 $a$ 后，如果之后继续按当前策略 $\pi$ 玩，长期平均回报是多少。若策略仍带有随机探索，SARSA 会把“之后可能随机按错”的风险也算进去，因此它评价的是现实中的实际玩法，而不是假设未来每步都完美选择的玩法。


## 2. SARSA 这个名字就是一次更新的五元组

SARSA 来自五个量的首字母：

$$
(S_t,A_t,R_{t+1},S_{t+1},A_{t+1}).
$$

它们依次表示：当前状态、当前实际动作、执行后的即时奖励、下一状态，以及在下一状态按**同一当前策略**实际选出的下一动作。最后这个 $A_{t+1}$ 是 SARSA 与 Q-learning 最关键的区别。


## 3. 用表格一步步学习

当状态和动作数量都有限时，可以画一张 $Q$ 表：行是状态，列是动作，格子 $Q(s,a)$ 是该状态动作的当前评分。开始时可全部填 $0$。之后不断重复：

1. 在 $s_t$ 按当前策略 $\pi$ 选 $a_t$；
2. 执行后观察 $r_{t+1}$ 和 $s_{t+1}$；
3. 在 $s_{t+1}$ 按同一策略实际选出 $a_{t+1}$；
4. 用这五个量更新表中的**一个格子** $Q(s_t,a_t)$；
5. 令当前状态动作变为 $(s_{t+1},a_{t+1})$，继续游戏。

SARSA 的 TD 目标为：

$$
y_t=r_{t+1}+\gamma Q(s_{t+1},a_{t+1}).
$$

用这个目标修正当前格子：

$$
Q(s_t,a_t)\leftarrow Q(s_t,a_t)+\alpha\left[y_t-Q(s_t,a_t)\right].
$$

同一个状态动作组合被经历得越多，评分通常越准确；若策略逐步改善，表格也会逐步追踪更好策略的价值。


## 4. 这是 TD，不是蒙特卡洛

SARSA 确实用一次实际经验来修正一个长期平均估计，但严格分类它是**时序差分（TD）方法**，不是蒙特卡洛方法。

| 方法 | 更新时用什么作为参考答案？ | 是否必须等待回合结束？ |
| --- | --- | --- |
| 蒙特卡洛 | 从当前时刻到终点的真实完整回报 | 是 |
| SARSA | 一步真实奖励 $+$ 下一状态下实际动作的当前评分 | 否 |

SARSA 走完一步、选出下一动作后就能更新；它把未知的很远未来，用 $Q(s_{t+1},a_{t+1})$ 的当前估计代替。终点附近的真实奖励会先让终点附近的格子变准，再逐步向前传播。一次经验样本参与更新的直觉和蒙特卡洛相近，但 SARSA 的目标不使用完整回合回报，因此它是 TD。


## 5. 与 Q-learning 的一个关键差别

两者的当前状态动作都来自经验，但下一状态的处理不同：

$$
\begin{aligned}
\text{Q-learning 目标}&=r_{t+1}+\gamma\max_aQ(s_{t+1},a),\\
\text{SARSA 目标}&=r_{t+1}+\gamma Q(s_{t+1},a_{t+1}).
\end{aligned}
$$

Q-learning 假设下一步一定选评分最高的动作；SARSA 使用策略实际选出的下一动作。若策略会以小概率随机探索，SARSA 会学习一条对探索风险也更稳妥的路线。Q-learning 是异策略方法，学习最优动作价值；SARSA 是同策略方法，学习当前策略价值。


## 6. 为什么标准 SARSA 不直接用经验回放，以及怎样做 Critic

SARSA 要评价的是当前策略 $\pi_{\mathrm{now}}$。但回放池里的旧轨迹由过去策略 $\pi_{\mathrm{old}}$ 产生；策略改变后，旧轨迹中下一动作的分布不再代表当前策略。因此标准 SARSA 不能像 Q-learning 一样，直接把旧四元组反复随机抽出训练。

在 Actor–Critic 中：Actor 是策略 $\pi_\theta$，负责选动作；Critic 是 $Q_w(s,a)$，负责评价 Actor 当前策略下动作的长期价值。若每一步的 $a_{t+1}$ 都由当前 Actor 实际选出，SARSA 的目标可训练 Critic：

$$
Q_w(s_t,a_t)\leftarrow r_{t+1}+\gamma Q_w(s_{t+1},a_{t+1}).
$$

Critic 不直接替 Actor 选动作；它告诉 Actor“照你当前这套真实玩法，这个动作好不好”。Actor 再根据 Critic 的评价提高好动作的选择概率。


## 7. 神经网络下：在线更新不等于经验回放

把表格换成价值网络 $q(s,a;w)$ 后，SARSA 的网络输入状态，输出各个离散动作的评分；训练仍然使用同一个实际下一动作 $a_{t+1}$：

$$
y_t=r_{t+1}+\gamma q(s_{t+1},a_{t+1};w).
$$

网络将当前预测 $q(s_t,a_t;w)$ 向目标 $y_t$ 靠近，通过反向传播更新参数 $w$。网络结构和 DQN 很像，真正不同的是 TD 目标：DQN 取下一状态所有动作中的最高评分；SARSA 取当前策略实际选出的下一动作评分。

要分清两件事：

- **在线更新**：做一步，得到新经验，马上训练一次。DQN 和 SARSA 都可以这样做。
- **经验回放**：把旧经验存入回放池，过一段时间后随机抽取、反复训练。标准 DQN 可以直接做；标准 SARSA 不可以直接做。

DQN 重放旧四元组 $(s,a,r,s')$ 时，使用当前网络重新计算：

$$
y_{\mathrm{DQN}}=r+\gamma\max_a q_{\mathrm{current}}(s',a).
$$

它不需要知道旧数据当年在 $s'$ 实际选了什么动作，也不依赖当年的行为策略。SARSA 的旧五元组中则有 $a'_{\mathrm{old}}\sim\pi_{\mathrm{old}}$；若当前策略已变为 $\pi_{\mathrm{now}}$，这个旧动作不再代表当前策略在 $s'$ 会怎么选。用它训练，会混淆“旧策略产生的数据”和“当前策略的价值”。

因此，标准神经网络 SARSA 通常按当前轨迹在线更新：先在 $s'$ 按当前策略选出 $a'$，再用它更新前一步。只有策略固定不变，或额外使用重要性采样等校正方法时，才可能谨慎利用旧数据；这已超出标准 SARSA。


## 8. 小结与自检

- SARSA 维护的是当前策略的动作价值 $Q_\pi(s,a)$。
- 一次更新使用 $(S,A,R,S',A')$，其中 $A'$ 是当前策略实际选出的下一动作。
- 它按一步 TD 目标更新表格，因此是 TD 方法，不是蒙特卡洛方法。
- 在 Actor–Critic 中，SARSA 可以训练评价当前 Actor 的 Critic。
- 在线更新不等于经验回放：DQN 可直接回放旧经验，标准 SARSA 不能直接回放旧策略产生的经验。

自检：在下一状态中，动作甲的评分为 $10$，动作乙的评分为 $2$。智能体按当前带探索策略实际选到了乙，当前奖励为 $1$，$\gamma=0.9$。SARSA 和 Q-learning 分别使用什么 TD 目标？

答：SARSA 使用 $1+0.9\times2=2.8$，因为实际选到乙；Q-learning 使用 $1+0.9\times10=10$，因为它取下一状态评分最高的动作甲。
